# **[Comparative Analysis] Ablation Study Synthesis (Herz-Jesus-P25)**

##### **The Objective: Comparative Evaluation**
The objective of this analysis is to synthesize the architectural impact of the Phase 2 ablation studies against the Phase 1 geometric baseline. By mapping the performance of each feature matching architecture (SIFT, SuperPoint, and ORB) across four distinct pipeline conditions (Baseline, No CLAHE, No Symmetric Matching, and No Parallax Filter), we can directly quantify the reliance of each algorithm on specific pre-processing, filtering, and geometric failsafes.

##### **The Evaluation Scope: Herz-Jesus-P25**
This specific analysis evaluates the **Herz-Jesus-P25** dataset. Scaling up to a perfectly calibrated 25-image sequence, this dataset introduces a significantly longer camera trajectory and a larger architectural scale. It serves as an intermediate stress test to evaluate how the removal of specific pipeline components exacerbates tracking fragmentation and geometric drift before tackling the severe perspective distortions of the later datasets.

#### **Setup & Data Extraction**

In [1]:
import os
import json
import pandas as pd
from pathlib import Path

In [2]:
EXPERIMENT_REPO_OWNER = "chuo-v"
EXPERIMENT_REPO_NAME = "incremental-sfm-experiments"

if 'google.colab' in str(get_ipython()):
    from google.colab import userdata

    if not os.path.exists(EXPERIMENT_REPO_NAME):
        try:
            token = userdata.get('GITHUB_TOKEN')
            repo_url = f"https://{token}@github.com/{EXPERIMENT_REPO_OWNER}/{EXPERIMENT_REPO_NAME}.git"
            print(f"Cloning private repository via PAT...")
        except userdata.SecretNotFoundError:
            repo_url = f"https://github.com/{EXPERIMENT_REPO_OWNER}/{EXPERIMENT_REPO_NAME}.git"
            print(f"Cloning public repository...")

        !git clone -q --depth 1 {repo_url} &> /dev/null

    os.chdir(EXPERIMENT_REPO_NAME)
    print(f"Environment Ready: Working directory set to {os.getcwd()}")
else:
    # Local Execution: Notebook is inside notebooks/03_comparative_analysis/
    # Move up two directories to reach the repository root
    repo_root = os.path.abspath("../../")
    os.chdir(repo_root)
    print(f"Environment Ready: Working directory set to {os.getcwd()}")

Cloning private repository via PAT...
Environment Ready: Working directory set to /content/incremental-sfm-experiments


In [3]:
repo_dir = Path.cwd()
dataset_name = "02_benchmark_herz_jesus_p25"
matchers = ["SIFT", "SuperPoint", "ORB"]
experiments = {
    "Baseline": repo_dir / "output" / "01_final" / dataset_name,
    "No CLAHE": repo_dir / "output" / "02_ablations" / "01_disable_clahe" / dataset_name,
    "No Symmetric Matching": repo_dir / "output" / "02_ablations" / "02_disable_symmetric_matching" / dataset_name,
    "No Parallax Filter": repo_dir / "output" / "02_ablations" / "03_disable_parallax_filter" / dataset_name
}

In [4]:
metrics_data = []

# Traverse the output directories and compile the JSON data
for exp_name, exp_path in experiments.items():
    for matcher in matchers:
        metrics_path = exp_path / "metrics" / f"run_metrics_{matcher}.json"

        if metrics_path.exists():
            with open(metrics_path, "r") as f:
                data = json.load(f)
                summary = data["results"]["summary"]

                metrics_data.append({
                    "Condition": exp_name,
                    "Matcher": matcher,
                    "Total 3D Points": summary.get("total_3d_points", 0),
                    "Overall RMSE (px)": round(summary.get("overall_rmse", 0.0), 3),
                    "Avg Frame Time (s)": round(summary.get("avg_frame_time_seconds", 0.0), 3),
                    "Mean Track Length": round(summary.get("mean_track_length", 0.0), 3),
                    "Tracking Inlier Ratio (%)": round(summary.get("tracking_inlier_ratio", 0.0) * 100, 1),
                    "Match Survival Rate (%)": round(summary.get("match_survival_rate", 0.0) * 100, 1)
                })
        else:
            print(f"Warning: Metrics not found for {matcher} under {exp_name} at {metrics_path}")

df = pd.DataFrame(metrics_data)

matcher_order = ["SIFT", "SuperPoint", "ORB"]
df['Matcher'] = pd.Categorical(df['Matcher'], categories=matcher_order, ordered=True)
condition_order = ["Baseline", "No CLAHE", "No Symmetric Matching", "No Parallax Filter"]
df['Condition'] = pd.Categorical(df['Condition'], categories=condition_order, ordered=True)
df = df.sort_values(['Matcher', 'Condition'])

display(df.style.hide(axis="index"))

Condition,Matcher,Total 3D Points,Overall RMSE (px),Avg Frame Time (s),Mean Track Length,Tracking Inlier Ratio (%),Match Survival Rate (%)
Baseline,SIFT,50867,1.444000,47.757000,2.636000,81.300000,38.500000
No CLAHE,SIFT,14234,1.765000,3.463000,2.510000,67.000000,28.600000
No Symmetric Matching,SIFT,55030,1.552000,26.134000,2.591000,75.800000,34.700000
No Parallax Filter,SIFT,51354,1.390000,70.347000,2.650000,82.900000,39.700000
Baseline,SuperPoint,30465,1.955000,5.207000,2.299000,32.200000,16.000000
No CLAHE,SuperPoint,26514,1.990000,4.384000,2.193000,23.100000,9.500000
No Symmetric Matching,SuperPoint,30465,1.955000,5.643000,2.299000,32.200000,16.000000
No Parallax Filter,SuperPoint,30647,1.955000,9.751000,2.297000,32.200000,16.000000
Baseline,ORB,63738,1.823000,14.621000,2.134000,57.700000,11.200000
No CLAHE,ORB,21588,1.656000,1.123000,2.236000,72.700000,19.600000


#### **Grouped Bar Chart Visualization**

In [5]:
%%capture
!pip install kaleido==0.2.1 plotly==5.24.1

In [6]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

In [7]:
colors = {
    "Baseline": "#1f77b4",               # Blue
    "No CLAHE": "#ff7f0e",               # Orange
    "No Symmetric Matching": "#d62728",  # Red
    "No Parallax Filter": "#2ca02c"      # Green
}

metrics_to_plot = [
    ("Total 3D Points", 1, 1),
    ("Overall RMSE (px)", 1, 2),
    ("Avg Frame Time (s)", 1, 3),
    ("Mean Track Length", 2, 1),
    ("Tracking Inlier Ratio (%)", 2, 2),
    ("Match Survival Rate (%)", 2, 3)
]

In [8]:
fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=(
        "Total 3D Points",
        "Overall RMSE",
        "Avg Frame Time",
        "Mean Track Length",
        "Tracking Inlier Ratio %",
        "Match Survival Rate %"
    ),
    vertical_spacing=0.15
)

# Populate grouped bars per metric
for condition in condition_order:
    cond_df = df[df["Condition"] == condition]
    if cond_df.empty:
        continue

    for metric, row, col in metrics_to_plot:
        show_legend = True if (row == 1 and col == 1) else False

        fig.add_trace(
            go.Bar(
                x=cond_df["Matcher"],
                y=cond_df[metric],
                name=condition,
                marker_color=colors[condition],
                showlegend=show_legend
            ),
            row=row, col=col
        )

fig.update_layout(
    title_text=f"Ablation Comparison Synthesis: {dataset_name.split('_', 2)[-1]}",
    barmode='group',
    width=1400,
    height=800,
    template="plotly_white",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.05,
        xanchor="right",
        x=1
    )
)

fig.show()

# Export synthesis figure
vis_dir = Path("output/03_comparative_analysis") / dataset_name
vis_dir.mkdir(parents=True, exist_ok=True)
synthesis_chart_path = vis_dir / f"metrics_synthesis_chart_{dataset_name}.png"
fig.write_image(str(synthesis_chart_path), scale=2)
print(f"\nSynthesis chart exported to: {synthesis_chart_path}")


Synthesis chart exported to: output/03_comparative_analysis/02_benchmark_herz_jesus_p25/metrics_synthesis_chart_02_benchmark_herz_jesus_p25.png
